# 02 · Topic windows sanity check
Clicks per topic over time: each topic's resources should be used mostly inside its window (shaded).

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
P = ROOT / 'data' / 'processed'; TAB = ROOT / 'results' / 'tables'; FIG = ROOT / 'results' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)


In [2]:
vd = pd.read_parquet(P/'vle_daily.parquet'); topics = pd.read_parquet(P/'topics.parquet')
pres = [('DDD','2013J'),('DDD','2014J'),('FFF','2013J'),('FFF','2014J')]
fig, axes = plt.subplots(len(pres),1, figsize=(10, 2.6*len(pres)), sharex=True)
cols = plt.cm.tab10.colors
for ax,(m,p) in zip(axes,pres):
    d = vd[(vd.code_module==m)&(vd.code_presentation==p)].groupby(['topic','date']).clicks.sum().reset_index()
    tp = topics[(topics.code_module==m)&(topics.code_presentation==p)]
    for _,r in tp.iterrows():
        k=int(r.topic); ax.axvspan(r.t_open, r.t_due, color=cols[k-1], alpha=0.08)
        dk = d[d.topic==k]; ax.plot(dk.date, dk.clicks, color=cols[k-1], lw=1, label=f'topic {k}')
    ax.set_title(f'{m} {p}: cohort clicks per topic per day (shaded = topic window)', fontsize=9); ax.set_ylabel('clicks')
axes[0].legend(ncol=6, fontsize=7); axes[-1].set_xlabel('day relative to module start')
fig.tight_layout(); fig.savefig(FIG/'topic_windows_clicks.png', dpi=150)

In [3]:
# share of each topic's clicks that fall inside its own window
vd = vd.merge(topics[['code_module','code_presentation','topic','t_open','t_due']], on=['code_module','code_presentation','topic'])
vd['inside'] = (vd.date>vd.t_open)&(vd.date<=vd.t_due)
vd['before'] = vd.date<=vd.t_open; vd['after'] = vd.date>vd.t_due
r = vd.groupby(['code_module','topic']).apply(lambda g: pd.Series({'inside':(g.clicks*g.inside).sum()/g.clicks.sum(),'before':(g.clicks*g.before).sum()/g.clicks.sum(),'after':(g.clicks*g.after).sum()/g.clicks.sum()}))
r.round(3)  # the same table is written by scripts/01_preprocess.py

C:\Users\Charv\AppData\Local\Temp\ipykernel_21624\4227231080.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  r = vd.groupby(['code_module','topic']).apply(lambda g: pd.Series({'inside':(g.clicks*g.inside).sum()/g.clicks.sum(),'before':(g.clicks*g.before).sum()/g.clicks.sum(),'after':(g.clicks*g.after).sum()/g.clicks.sum()}))


inside  before  after
code_module topic                       
DDD         1       0.446   0.000  0.554
            2       0.679   0.167  0.154
            3       0.673   0.142  0.184
            4       0.624   0.173  0.204
            5       0.731   0.136  0.133
            6       0.332   0.173  0.495
FFF         1       0.468   0.000  0.532
            2       0.523   0.165  0.312
            3       0.494   0.076  0.430
            4       0.503   0.053  0.444
            5       0.602   0.087  0.310
            6       0.688   0.310  0.002